# Import FP&A budget and rolling forecast into Azure SQL

Load `Data/budget_detail.csv` and `Data/forecast_detail.csv` into `silver.fact_plan`. Publish Gold views for Actual vs Budget and Actual vs Forecast at **month + account + scenario + version** grain. Run `99_run_pipeline.ipynb` first to populate `silver.dim_account` and `gold.pnl_monthly_actual`.

This notebook uses Azure SQL only; it does not write to QuickBooks. The import is disabled by default. CSV validation is offline; account-mapping preview reads Azure SQL.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "planning.py").exists():
    raise FileNotFoundError("Open from the project root or notebooks folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.azure_sql import get_engine
from src.planning import load_plans, map_accounts, import_plans

## Configure versions and calendar

The supplied files each cover September 2023 through August 2026, not just one year. With the September fiscal start below, budgets are labeled FY2024, FY2025, and FY2026 (ending-year convention). Change this setting if your FP&A calendar differs.

Forecast version names come directly from the file. The explicit as-of mapping separates the snapshot date from the month being forecast. The current version contains Actualized rows through April 2026 and Forecast rows for May-August 2026. Add a mapping entry for each new forecast version; the month containing the as-of date is treated as closed/actualized.

In [ ]:
DATA_DIR = PROJECT_ROOT / "Data"
BUDGET_LABEL = "Original Budget"
FISCAL_START_MONTH = 9
FORECAST_AS_OF = {"2026-04 Latest Forecast": "2026-04-30"}
APPLY_IMPORT = True

In [ ]:
plans = load_plans(
    DATA_DIR, budget_label=BUDGET_LABEL,
    fiscal_start_month=FISCAL_START_MONTH, forecast_as_of=FORECAST_AS_OF,
)
display(plans.groupby(["scenario", "version_name", "basis"]).agg(
    rows=("account_number", "size"), first_month=("month_start", "min"),
    last_month=("month_start", "max"), reporting_total_usd=("reporting_amount_usd", "sum")
).reset_index())
display(plans.head())
print("CSV validation complete. No database writes.")

## Verify QBO account mapping (read-only)

Planning account numbers must map uniquely to `silver.dim_account.account_number`. The QBO account ID is stored on the plan rows, so Actual and Plan share the same Power BI account dimension. Missing or ambiguous accounts and conflicting P&L classifications block the import.

In [ ]:
engine = get_engine()
with engine.connect() as connection:
    target = connection.execute(text("SELECT DB_NAME() AS database_name, @@SERVERNAME AS server_name")).mappings().one()
    accounts = pd.read_sql(text("SELECT account_id, account_number, classification FROM silver.dim_account"), connection)
    mapped_preview = map_accounts(plans, accounts)
print("Target:", dict(target))
display(mapped_preview[["account_number", "account_name", "account_id"]].drop_duplicates())

## Import and publish comparison views

The transaction reloads the **complete versions present in these files**, preserving other versions already loaded. Rerunning the same files does not duplicate rows. Use a new budget label or Forecast_Version to retain a distinct revision. Supply full snapshots for each version, not incremental month patches.

The source amount and reporting amount are stored as DECIMAL(18,2), with the file hash, load batch, and UTC load time. Before commit, the loader reconciles row counts and reporting totals by version. Any failure rolls back both imports and view changes.

The existing Actuals table is not modified. Its expense-positive values are converted to expense-negative reporting amounts in the comparison view.

In [ ]:
load_result = None
if not APPLY_IMPORT:
    print("Preview only. Set APPLY_IMPORT = True and rerun settings and this cell to load.")
else:
    plans = load_plans(DATA_DIR, budget_label=BUDGET_LABEL,
                       fiscal_start_month=FISCAL_START_MONTH, forecast_as_of=FORECAST_AS_OF)
    with engine.begin() as connection:
        load_result = import_plans(connection, plans)
    display(load_result)
    print("Committed budget/forecast import and Gold comparison views.")

In [ ]:
if load_result is not None:
    with engine.connect() as connection:
        display(pd.read_sql(text("""
            SELECT scenario, version_name, basis, COUNT_BIG(*) AS rows,
                   SUM(reporting_amount_usd) AS reporting_total_usd
            FROM gold.fpa_monthly
            GROUP BY scenario, version_name, basis
            ORDER BY scenario, version_name, basis
        """), connection))

## Power BI model

Import these Gold objects and create single-direction, one-to-many relationships from each dimension to `fpa_monthly`:

| Dimension | Relationship key | Purpose |
|---|---|---|
| `gold.fpa_account` | `account_id` | Account, account number, P&L section and subcategory |
| `gold.fpa_month` | `month_start` | Calendar/fiscal period filtering and sorting |
| `gold.fpa_version` | `version_key` | Scenario and version selection |

`gold.fpa_monthly` is the comparison fact. Budget and forecast department rows are aggregated to month/account because the existing Actuals Gold table has no department dimension. `silver.fact_plan` retains department and budget driver detail for plan-only analysis. Do not compare departmental plans to undivided company Actuals.

Use `reporting_amount_usd` for additive P&L comparisons: income is positive; expenses are negative. Actual minus Budget (or Forecast) is therefore a favorable-positive variance. Preserve negative contra-revenue values. `amount_usd` keeps the original expense-positive presentation.

Filter measures by scenario. Select exactly one forecast version; never sum several forecast snapshots. For budgets, choose one budget revision per fiscal year. A version-dimension slicer can filter Actual out of the fact, so Actual measures should remove version-dimension filters before applying Scenario = Actual. Separate Budget and Forecast selectors are useful when building the dashboard.

For a full-year rolling forecast, include both Actualized and Forecast rows **from that snapshot**. For future-period comparisons, use Basis = Forecast and apply the same month selection to Actual. Do not add the snapshot's Actualized rows to live Actuals: that double-counts history. Actualized snapshot values may differ from later accounting restatements.

The month dimension contains the months present in Actual or Plan; use a separate daily calendar when implementing daily time-intelligence measures. Refresh Power BI after imports or Actuals refreshes. Non-schema-bound comparison views continue reading the latest Actuals table after a completed pipeline rebuild.

In [ ]:
engine.dispose()